In [7]:
#%pip install pandas lxml html5lib --quiet
import pandas as pd
url = "https://www.bbc.co.uk/sport/football/world-cup/table"
try:
    tables = pd.read_html(url)
    
    for index, group_df in enumerate(tables[:-1]):
        #print(f"\n--- Group {chr(65 + index)} Live Table ---")
        
        group_df.columns = [str(col).strip() for col in group_df.columns]
        
        group_df['Team'] = group_df['Team'].str.replace(r'^\d+\.?\s*', '', regex=True)

        #print(group_df[['Team', 'Played', 'Points']])

except Exception as e:
    print("Error fetching live data")

In [8]:
family = {'Eagle 1' : ['Brazil', 'Turkey', 'Ivory Coast', 'South Korea', 'Paraguay', 'United States', 'Austria', 'Portugal'], 
          'Eagle 2' : ['Belgium', 'Australia', 'France', 'Algeria', 'Morocco', 'Canada', 'Czech Republic', 'Iraq'], 
          'Eagle 3' : ['Panama', 'Ecuador', 'Argentina', 'Japan', 'England', 'Ghana', 'Iran', 'South Africa'], 
          'Eagle 4' : ['Qatar', 'Germany', 'Senegal', 'Colombia', 'Curaçao', 'Switzerland', 'Congo DR', 'Netherlands'], 
          'Eagle 5' : ['Scotland', 'Saudi Arabia', 'Tunisia', 'Bosnia-Herzegovina', 'Haiti', 'Mexico', 'Croatia', 'Egypt'], 
          'Eagle 6' : ['Jordan', 'Cape Verde', 'Norway', 'Spain', 'Uruguay', 'Sweden', 'Uzbekistan', 'New Zealand'],}

In [13]:
all_teams = []
for group_df in tables[:-1]:
    group_df.columns = [str(col).strip() for col in group_df.columns]
    group_df['Team'] = group_df['Team'].str.replace(r'^\d+\.?\s*', '', regex=True)
    all_teams.append(group_df[['Team', 'Played', 'Goal Difference', 'Points']])

all_teams_df = pd.concat(all_teams, ignore_index=True)

team_to_member = {team: member for member, teams in family.items() for team in teams}

all_teams_df['Member'] = all_teams_df['Team'].map(team_to_member)

family_table = all_teams_df.groupby('Member')[['Played', 'Goal Difference', 'Points']].sum()

family_table['Points per match'] = round(family_table['Points'] / family_table['Played'], 2)

family_table = family_table.sort_values('Points', ascending =False)

print(family_table)

         Played  Goal Difference  Points  Points per match
Member                                                    
Eagle 1       6                0      10              1.67
Eagle 5       5               -2       7              1.40
Eagle 4       5                0       6              1.20
Eagle 2       4                1       5              1.25
Eagle 6       1                4       3              3.00
Eagle 3       3               -3       1              0.33


In [17]:
from IPython.display import HTML
import json
 
# ── build JSON from your dataframe ──────────────────────────────────────────
rows = []
for member, row in family_table.iterrows():
    rows.append({
        "Member": member,
        "Played": int(row["Played"]),
        "GD":     int(row["Goal Difference"]),
        "Points": int(row["Points"]),
        "PPM":    float(row["Points per match"])
    })
 
data_json = json.dumps(rows)
 
# ── widget ───────────────────────────────────────────────────────────────────
html = """
<style>
table{width:100%;border-collapse:collapse;font-size:14px;font-family:sans-serif}
th,td{padding:10px 14px;text-align:left;border-bottom:1px solid #e5e5e5}
td:not(:first-child),th:not(:first-child){text-align:right}
thead th{font-weight:500;font-size:13px;color:#888;white-space:nowrap}
tbody tr:hover td{background:#f9f9f9}
.sort-btn{display:inline-flex;align-items:center;gap:6px;padding:6px 14px;
          border:1px solid #ddd;border-radius:6px;background:#fff;
          font-size:13px;cursor:pointer;margin-right:8px;margin-bottom:16px}
.sort-btn.active{background:#f0f0f0;border-color:#bbb;font-weight:500}
.rank{color:#aaa;font-size:12px}
.badge{background:#f3f3f3;border-radius:4px;padding:2px 7px;font-size:12px;color:#666}
.pos{color:green}.neg{color:#c0392b}
</style>
 
<div>
  <button class="sort-btn active" id="btn-pts" onclick="sortBy('Points')">&#127942; Sort by Points <span id="icon-pts">&#9660;</span></button>
  <button class="sort-btn" id="btn-ppm" onclick="sortBy('PPM')">&#8709; Sort by PPM <span id="icon-ppm">&#9660;</span></button>
</div>
 
<table>
  <thead>
    <tr>
      <th>#</th><th style="text-align:left">Member</th>
      <th>Played</th><th>GD</th><th>Points</th><th>PPM</th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>
 
<script>
const rawData = DATA_JSON_PLACEHOLDER;
let currentSort = 'Points', desc = true;

function sortBy(key) {
  if (currentSort === key) desc = !desc;
  else { currentSort = key; desc = true; }
  render();
}
 
function render() {
  const sorted = [...rawData].sort((a, b) => {
    const va = currentSort === 'Points' ? a.Points : a.PPM;
    const vb = currentSort === 'Points' ? b.Points : b.PPM;
    return desc ? vb - va : va - vb;
  });
 
  ['pts', 'ppm'].forEach(k => {
    const key = k === 'pts' ? 'Points' : 'PPM';
    document.getElementById('btn-' + k).classList.toggle('active', currentSort === key);
    document.getElementById('icon-' + k).innerHTML =
      currentSort === key ? (desc ? '&#9660;' : '&#9650;') : '&#9660;';
  });
 
  document.getElementById('tbody').innerHTML = sorted.map((r, i) => {
    const gdSign = r.GD >= 0 ? '+' : '';
    const gdClass = r.GD >= 0 ? 'pos' : 'neg';
    const ptsBold = currentSort === 'Points' ? '600' : '400';
    const ppmBold = currentSort === 'PPM' ? '600' : '400';
    return `<tr>
      <td><span class="rank">${i + 1}</span></td>
      <td style="font-weight:500">${r.Member}</td>
      <td><span class="badge">${r.Played}</span></td>
      <td class="${gdClass}">${gdSign}${r.GD}</td>
      <td style="font-weight:${ptsBold}">${r.Points}</td>
      <td style="font-weight:${ppmBold}">${r.PPM.toFixed(2)}</td>
    </tr>`;
  }).join('');
}
 
render();
</script>
""".replace("DATA_JSON_PLACEHOLDER", data_json)
 
HTML(html)

#,Member,Played,GD,Points,PPM
